In [21]:
#import bibliotek i wczytanie danych
import pandas as pd

import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import SGDClassifier
from scipy.sparse import hstack

import numpy as np

import joblib
import os

df = pd.read_csv('english_bg3_reviews_cleaned.csv', delimiter=';', encoding='utf-8')
df.head()

,review_id,review,voted_up,author_steamid,date_str,review_length,review_word_count,clean_review
0,209492575,very good game :),True,76561198079322180,2025-11-17 20:49:42,17,4,very good game
1,209492181,"Yes, it is as good as everyone says it is. Buy...",True,76561198451366797,2025-11-17 20:43:50,50,12,yes it is as good as everyone says it is buy it
2,209491639,gud,True,76561198259769014,2025-11-17 20:35:33,3,1,gud
3,209490463,The greatest game of all time. It simply is. \r\n,True,76561198842852247,2025-11-17 20:17:24,47,9,the greatest game of all time it simply is
4,209488275,The dude trying to bite me at night was messed...,True,76561198335095296,2025-11-17 19:45:37,59,12,the dude trying to bite me at night was messed...


In [22]:
#przygotowanie ramki danych do trenowania modelu filtru spamu
df['clean_review'] = df['clean_review'].fillna("")


# funkcja definiująca treści spamowe
def spam_label(text):
    text = str(text).lower().strip()
    
    # wpisy które są zbyt krótkie
    if len(text) < 15: return 1 
    # powtórzenia znaków
    if len(set(text)) < 4 and len(text) > 10: return 1
    # zbyt długie niemerytoryczne słowa
    longest_word = max([len(w) for w in text.split()], default=0)
    if longest_word > 40: return 1
    
    return 0 # zero nie jest spamem

df['is_spam_label'] = df['clean_review'].apply(spam_label)

X = df['clean_review']
y = df['is_spam_label']


In [23]:
#tworzenie cech jakościowych 
df['review_length'] = df['clean_review'].apply(len)
df['review_word_count'] = df['clean_review'].apply(lambda x: len(x.split()))
df['num_emojis'] = df['clean_review'].apply(lambda x: len(re.findall(r'[^\w\s,]', str(x))))


In [ ]:
#wektory tf-idf
vectorizer = TfidfVectorizer(min_df=5, max_df=0.9, ngram_range=(1,2))
X_text = vectorizer.fit_transform(df['clean_review'])

In [25]:
#łączymy cechy tekstowe z dodatkami
X_additional = df[['review_length', 'review_word_count', 'num_emojis']].values
X = hstack([X_text, X_additional])
y = df['is_spam_label']

In [26]:
#podział danych na zbiór treningowy i testowy 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [27]:
#model 
model = SGDClassifier(loss='hinge', class_weight='balanced', max_iter=10000, random_state=42)
model.fit(X_train, y_train)

SGDClassifier(class_weight='balanced', max_iter=10000, random_state=42)

In [28]:
#ocena modelu
y_pred = model.predict(X_test)
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       1.00      0.86      0.92     14325
           1       0.49      0.97      0.65      1993

    accuracy                           0.87     16318
   macro avg       0.74      0.92      0.79     16318
weighted avg       0.93      0.87      0.89     16318



In [29]:
#SGDClassifier nie radzi sobie ze skrajnie nierzównoważonymi klasami
# Klasa False jest bardzo rzadka, a model przewiduje tylko klasę True
# accuracy jest wysoka, ale precision, recall i f1-score dla klasy False są zerowe
# optymalizacja modelu

from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler



# normalizacja cech dodatkowych i ograniczenie liczby cech tf-idf
scaler = StandardScaler(with_mean=False)  # sparse matrix nie może mieć dense mean centering
X_add_scaled = scaler.fit_transform(df[['review_length', 'review_word_count']])

df['clean_review'] = df['clean_review'].fillna('')
vectorizer = TfidfVectorizer(min_df=5, max_df=0.9, ngram_range=(1,2), max_features=50000)
X_text = vectorizer.fit_transform(df['clean_review'])
X = hstack([X_text, X_add_scaled])

df['low_quality'] = df['clean_review'].apply(lambda x: 1 if len(x.split()) <= 5 else 0)
y = df['low_quality']

# oversampling
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)


#podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

#trenowanie modelu po optymalizacji
model = SGDClassifier(loss='hinge', class_weight='balanced', max_iter=10000, random_state=42)
model.fit(X_train, y_train)

#ocena
y_pred = model.predict(X_test)
print("=== CLASSIFICATION REPORT (OPTIMIZED) ===")
print(classification_report(y_test, y_pred))


=== CLASSIFICATION REPORT (OPTIMIZED) ===
              precision    recall  f1-score   support

           0       1.00      0.94      0.97     12038
           1       0.94      1.00      0.97     12037

    accuracy                           0.97     24075
   macro avg       0.97      0.97      0.97     24075
weighted avg       0.97      0.97      0.97     24075



In [30]:
X_text_full = vectorizer.transform(df['clean_review'])

X_add_full = df[['review_length', 'review_word_count']].copy()

scaler = StandardScaler(with_mean=False)
X_add_scaled_full = scaler.fit_transform(X_add_full) 

X_full = hstack([X_text_full, X_add_scaled_full])

df['is_spam'] = model.predict(X_full)
spam_free_df = df[df['is_spam'] == 1].copy()


In [31]:
print(spam_free_df.head())        
print(df[df['is_spam']==1].head()) 
print(df["is_spam"].value_counts())


    review_id             review  voted_up     author_steamid  \
0   209492575  very good game :)      True  76561198079322180   
2   209491639                gud      True  76561198259769014   
7   209484454          good game      True  76561199237561360   
8   209484065  based dnd game :3      True  76561199492777926   
14  209478288     love this game      True  76561199013364740   

               date_str  review_length  review_word_count     clean_review  \
0   2025-11-17 20:49:42             15                  3  very good game    
2   2025-11-17 20:35:33              3                  1              gud   
7   2025-11-17 18:47:25              9                  2        good game   
8   2025-11-17 18:41:33             15                  3  based dnd game    
14  2025-11-17 17:14:35             14                  3   love this game   

    is_spam_label  num_emojis  low_quality  is_spam  
0               1           0            1        1  
2               1           0   

In [32]:
# podgląd odrzucanych recenzji
spam_examples = df[df['is_spam'] == 1]['clean_review'].head(20)
for review in spam_examples:
    print(review)


very good game 
gud
good game
based dnd game 
love this game
sword meets sorcery
good
its a way of life
goat
the best

bra
simply amazing
best game ever
murderhobo
pretty good
game aight
best game ever probably
karlack shadowheart laezel minthara and orin
goty


In [33]:
# filtr jest zbyt rygorystyczny, usuwa zbyt wiele recenzji
# należy dalej eksperymentować z cechami i modelem
# odrzucił np. "definitely one of the best games ive played in the last years"
# ustawimy inny próg żeby poluzować filtrację

In [34]:
# threshold
scores = model.decision_function(X)
threshold = -0.5
df['is_spam'] = (scores > threshold).astype(int)

print(df["is_spam"].value_counts())

spam_examples = df[df['is_spam'] == 1]['clean_review'].head(20)
for review in spam_examples:
    print(review)


is_spam
0    53740
1    27850
Name: count, dtype: int64
very good game 
gud
good game
based dnd game 
love this game
sword meets sorcery
good
its a way of life
goat
larian studios did it again and better
the best

bra
simply amazing
best game ever
murderhobo
pretty good
game aight
best game ever probably
karlack shadowheart laezel minthara and orin


In [35]:
# threshold 2
scores = model.decision_function(X)
threshold = -3.0
df['is_spam'] = (scores > threshold).astype(int)

print(df["is_spam"].value_counts())

spam_examples = df[df['is_spam'] == 1]['clean_review'].head(20)
for review in spam_examples:
    print(review)

is_spam
1    42647
0    38943
Name: count, dtype: int64
very good game 
yes it is as good as everyone says it is buy it
gud
the greatest game of all time it simply is 
the dude trying to bite me at night was messed up lfmao
one of the best rpgs of the last decade
good game
based dnd game 
if fuck around and find out was a game this is it
love this game
sword meets sorcery
good
its a way of life
goat
larian studios did it again and better
the best

great game like dnd but on a pc
bra
simply amazing


In [36]:
# threshold nic nie zmienia, filtr dalej odrzuca wartościowe recenzje
# problem nie leży w thresholdzie, a w cechach i modelu
# w takim razie trzeba usunąć cechy ilościowe, oversampling i skupić się tylko na tekście

In [ ]:
# bez cech ilościowych
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.95,
    ngram_range=(1,2),
    max_features=60000
)

X = vectorizer.fit_transform(df["clean_review"])
y = df["is_spam_label"]   

# zmiana modelu na Logistic Regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=5000,
    n_jobs=-1
)

model.fit(X, y)


scores = model.decision_function(X)

In [38]:
# threshold
threshold = -1.5

df["is_spam"] = (scores > threshold).astype(int)

print(df["is_spam"].value_counts())

spam_examples = df[df['is_spam'] == 1]['clean_review'].head(20)
for review in spam_examples:
    print(review)

is_spam
0    53048
1    28542
Name: count, dtype: int64
very good game 
yes it is as good as everyone says it is buy it
gud
good game
based dnd game 
love this game
sword meets sorcery
good
its a way of life
goat
the best

great game like dnd but on a pc
bra
simply amazing
best game ever
murderhobo
pretty good
game aight
best game ever probably


In [39]:
#zapis do plików csv
spam_free_df = df[df["is_spam"] == 0].copy() #nie spam
spam_df      = df[df["is_spam"] == 1].copy() #spam

spam_free_df.to_csv("english_bg3_reviews_no_spam.csv", sep=";", index=False, encoding='utf-8')
spam_df.to_csv("english_bg3_reviews_spam.csv", sep=";", index=False, encoding='utf-8')

In [ ]:
# zapis wytrenowanych modeli i wektorów w celu wykorzystania ich w aplikacji

if not os.path.exists('models'):
    os.makedirs('models')

# zapis wektoryzacji do spamu
joblib.dump(vectorizer, 'models/spam_vectorizer.pkl')

# zapis regresji logistycznej do spamu
joblib.dump(model, 'models/spam_model.pkl')

print("Pliki zapisane w folderze")

Zapisywanie modelu i wektoryzatora...
Pliki zapisane w folderze.
